# [FBNet](https://arxiv.org/pdf/1812.03443.pdf) on MNIST

In this exercise we will have fun with a simplified version of [FBNet](https://arxiv.org/pdf/1812.03443.pdf) algorithm for NAS. We will implement the following modules that were used in [FBNet](https://arxiv.org/pdf/1812.03443.pdf) architecture:

- a Depthwise separable convolution module,
- a FBNet Block,
- a NAS Experiment with a varying degree latency regularization.

We will finalize this exercise with a visualization of a [Pareto front](https://en.wikipedia.org/wiki/Pareto_front) plot for NAS experiments.

As a data we will use a standard MNIST dataset:

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

!pip install lightning


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 128

MNIST_SIZE = 28


train_dataset = torchvision.datasets.MNIST(
    root=r'./mnist/',
    train=True,
    transform=transforms.ToTensor(),
    download=True,
)

val_dataset = torchvision.datasets.MNIST(
    root=r'./mnist/',
    train=False,
    transform=transforms.ToTensor(),
    download=True,
)

train_x = 2 * (train_dataset.data.float() / 255. - 0.5)
val_x = 2 * (val_dataset.data.float() / 255. - 0.5)

train_y = train_dataset.targets
val_y = val_dataset.targets

train_dataset = torch.utils.data.TensorDataset(train_x.view(-1, 1, 28, 28), train_y)
val_dataset = torch.utils.data.TensorDataset(val_x.view(-1, 1, 28, 28), val_y)


train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True)

## Depthwise separable convolution

In this task you will implement a [depthwise separable convolution](https://arxiv.org/pdf/1902.00927v2.pdf) module. The key idea behind this module is to decompose a standard convolution into two consecutive convolutions that decrease the amount of computations:

- a depthwise convolution (with `groups=in_channels`, and  `out_channels=in_channels`, with appropriate `kernel_size` and `padding`),
- a pointwise convolution (with a 1x1 filter size and `out_channels`).

**Hint:** [This](https://discuss.pytorch.org/t/how-to-modify-a-conv2d-to-depthwise-separable-convolution/15843) forum post might serve as a nice help in your implementation.

In [ ]:
class DepthwiseSeparableConnvolution(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        padding,
    ):
        super(DepthwiseSeparableConnvolution, self).__init__()
        # TODO

    def forward(self, x):
        # TODO


## FBNet Block

Now it is time to implement FBNet Block. In its constructor it should initiate:

* a module list with 7 operations (each module should have the same `in_channels` and `out_channels` parameters and and a padding preserving the spatial dimensions):
    - Standard convolution with 1x1 filters,
    - Standard convolution with 3x3 filters,
    - Standard convolution with 5x5 filters,
    - Standard convolution with 7x7 filters,
    - Deptwise separable convolution with 3x3 filters,
    - Deptwise separable convolution with 5x5 filters,
    - Deptwise separable convolution with 7x7 filters ,
* a `logits` tensor of size `(7,)` initiated with zeros that are used to compute the preference of each of the submodules above,
* a `latencies` optional tensor initally set to `None`.

**Caution:** Be sure that your module list store these submodules in the presented order as this order is later used in the visualization purposes.

The constructor should additionally accept a `profiling_evals_nb` - a parameter that determines the number of module evaluation during the profiling method described below:

###`profile` method:

`FBNetModule` should have a `profile` method that given an input tensor `x` will run a `profiling_evals_nb` of evaluation of each of its submodules on this tensor, register their execution times and set a latency of each submodule to a median from its evaluations. It should set a `latencies` tensor to a `Tensor` of a shape `(7,)` with values of the latencies computed for each of the submodules.

###`forward` method:

It should accept an input tensor `x`, a temperature parameter `tau`, and a distilation flag `distil` and return `output` and `latency` tensors using the following instructions:

- in case when `distil == False` it should:
     1. Compute [`gumbel_softmax`](https://pytorch.org/docs/stable/generated/torch.nn.functional.gumbel_softmax.html) weights `m` (see Equation 8 from an original [paper](https://arxiv.org/pdf/1812.03443.pdf)) with temperature `tau`,
     2. Compute the output of each of submodules on tensor `x` and average these outputs with weights `m` to obtain the `output` tensor,
     3. Compute the average latency by averaging `latency` tensor with weights `m` (see Equation 9 from an original [paper](https://arxiv.org/pdf/1812.03443.pdf)) to obtain `latency` tensor.
- in case when `distil == True` it should select the submodule with the highest `logits` value and output its `latency` and the `output` tensor obtained by applying seleced submodule to the tensor `x`.

**Question:** Why every submodule in this block should have the same `out_channels`?

In [ ]:
import time


FB_NET_MODULES_NAMES = [
    'Conv\n1x1',
    'Conv\n3x3',
    'Conv\n5x5',
    'Conv\n7x7',
    'DConv\n3x3',
    'DConv\n5x5',
    'DConv\n7x7'
]


import numpy

class FBNetModule(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        profiling_evals_nb=50,
    ):
        super(FBNetModule, self).__init__()
        self.profiled = False
        self.profiling_evals_nb = profiling_evals_nb
        self.fb_modules = nn.ModuleList(
            [
                # TODO
            ]
        )
        self.logits = nn.Parameter(torch.zeros(size=(7,)))
        self.latencies = None

    def forward(self, x, tau=1.0, distil=False):
        if not self.profiled:
            self.profile(x)
        # TODO

    def profile(self, x):
        # TODO
        self.profiled = True
        return self.latencies

## FBNetModel

In this part we will implement a [`lightning`](https://www.pytorchlightning.ai/index.html) module consisting of three consecutive `FBNetModule`s. Your task is to implement:

- a `forward` method, that accepts input tensor `x`, a temperature parameter `tau` and a distiliation flag `distil`. It outputs a tensor `output` that is a result of application of a sequential application of three `FBNetModule` modules to an input tensor `x`. Application of each module should be followed by a `relu` activation. Additionally - you should apply a `pooling` operation after first two `FBNetModule`s that downsizes each spatial dimension by two. It should also output a `latency` tensor that is a sum of latencies from each of the `FBNetModule` modules,
- a `training_step` where you should:
     - compute the loss of your model that is a sum of [`CrossEntropyLoss`](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) and `log(latency)` multiplied by a weight `latency_weight`,
     - apply a temperature decay where after each training step you decrease the temperature `tau` by a factor of `1 - 1e-3`. An initial value of a `tau` parameter should be set to `1.0`.
     - log `accuracy` and `latency` after each step.
- a `validation` step - where you should validate your distilled (`distil` flag set to `True`) model by computing and logging its `accuracy` and `latency`.

In [ ]:
import lightning.pytorch as pl


FB_NET_BLOCK_NAMES = ['FBNetBlock 1', 'FBNetBlock 2', 'FBNetBlock 3']


class Model(pl.LightningModule):
    def __init__(
        self,
        latency_weight=1.0,
    ):
        # DO NOT CHANGE
        super(Model, self).__init__()
        self.pool = torch.nn.MaxPool2d(2, stride=2)
        self.first_block = FBNetModule(in_channels=1, out_channels=16)
        self.second_block = FBNetModule(in_channels=16, out_channels=32)
        self.third_block = FBNetModule(in_channels=32, out_channels=64)
        self.dense = nn.Linear(64 * 7 * 7, 10)
        self.temperature = 1.0 # USE THIS TENSOR AS YOUR TEMPERATURE
        self.latency_weight = latency_weight
        self.distil = False
        # YOU MAY EXTEND THE CONSTRUCTOR BELOW
        # HERE


    def training_step(self, input):
        # TODO

    def validation_step(self, input, batch_idx):
        # TODO

    def forward(self, x, tau=1.0, distil=None):
        distil = distil if distil is None else self.distil
        # TODO

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=1e-3)

    def get_fb_net_logits(self):
        return self.first_block.logits, self.second_block.logits, self.third_block.logits

    def plot_logits(self):
        logits = torch.softmax(torch.vstack(self.get_fb_net_logits()), -1).cpu().detach().numpy()
        plt.imshow(logits)
        for x in range(logits.shape[0]):
            for y in range(logits.shape[1]):
                plt.text(y - 0.25, x, round(logits[x, y], 2))
        plt.yticks(
            list(range(len(FB_NET_BLOCK_NAMES))),
            FB_NET_BLOCK_NAMES
        )
        plt.xticks(
            list(range(len(FB_NET_MODULES_NAMES))),
            FB_NET_MODULES_NAMES,
        )
        plt.show()

You can use the code below to debug your model. It trains model for 10 epochs, where for the first 5 epochs we train architecture (by training the logits of the `FBNetModule` modules) and later we only train the distilled, best model. We additionally plot architecture logits used for distilation so you can see which operations were selected.

In [ ]:
model = Model()

In [ ]:
from matplotlib import pyplot as plt
from lightning.pytorch.callbacks import Callback


class FBNetCallback(Callback):

    def __init__(self, distill_after=5):
        self.distill_after = distill_after

    def on_train_start(self, trainer, pl_module):
        pl_module.temperature = 1.0

    def on_train_epoch_end(self, trainer, pl_module):

        if pl_module.current_epoch == (self.distill_after - 1):
            print('Starting distillation')
            pl_module.distil = True
            pl_module.plot_logits()

In [ ]:
trainer = pl.Trainer(
        max_epochs=10,
        callbacks=FBNetCallback(),
)

In [ ]:
trainer.fit(model, train_dataloader)

In [ ]:
trainer.validate(dataloaders=val_dataloader)

### Pareto front

Use the code below to compute the Pareto front of your NAS. This code will compute `NB_OF_TRIES` models for each of preselected `LATENCY_WEIGHTS` and plot their `accuracy` and `latency`. Which regions of the presented plot might be considered as the best?

Questions:
- does the `latency_weight` influence the final latency of your model?
- does it affect the `accuracy`?
- which operations are selected for each of the `latency_weight`?
- does the `DepthwiseSeparableConvolution` looks like optimal choice for a fast computations in `pytorch`?
- is the architecture selected stable for a given `latency_weight`? If not - you may witness the infamous [Matthew effect](https://en.wikipedia.org/wiki/Matthew_effect#:~:text=The%20Matthew%20effect%20of%20accumulated,and%20the%20poor%20get%20poorer%22.). What's your intuition why the optimization might converge to different (and sometimes couterintuitive) solutions?

In [ ]:
from collections import defaultdict

LATENCY_WEIGHTS = [-1.0, 0.0, 1.0, 10.0]
NB_OF_TRIES = 2


pareto_front_results = defaultdict(list)


for latency_weight in LATENCY_WEIGHTS:
    for try_ in range(NB_OF_TRIES):
         print(f'LATENCY WEIGHT = {latency_weight}, TRY = {try_}')
         current_model = Model(latency_weight=latency_weight)
         current_trainer = pl.Trainer(
             max_epochs=10,
             callbacks=FBNetCallback(),
         )
         current_trainer.fit(current_model, train_dataloader)
         pareto_front_results[latency_weight].append(current_trainer.validate(dataloaders=val_dataloader))

In [ ]:
def visualize_pareto_front(pareto_front_results):
    for ind_, latency_weight in enumerate(pareto_front_results):
        for try_results in pareto_front_results[latency_weight]:
            plt.scatter(
                x=[try_results[0]['acc']],
                y=[try_results[0]['latency']],
                c=[plt.cm.tab20(ind_)],
                label=latency_weight,
            )
    plt.legend()
    plt.xlabel('Accuracy')
    plt.ylabel('Latency')

In [ ]:
visualize_pareto_front(pareto_front_results)